# CONIFER benchmark: design-based small-area estimation, known truth

A short, self-contained, reproducible benchmark of `conifer.DiameterDistribution` (v0.2) against the
design-direct estimator and a k-NN baseline, on a synthetic forest population with a **known truth**.
It reproduces the paper's headline pattern on a small scale in a few seconds:

1. **CONIFER wins in the data-poor (sparse) regime** and converges to the design-direct estimate as plots accumulate (the v0.2 Fay-Herriot consistency fix).
2. **The design-aware conformal set is calibrated** where the naive Gaussian interval under-covers.

Requires `pip install conifer-sae` (>= 0.2.0).

In [ ]:
import numpy as np, conifer
import matplotlib.pyplot as plt
print('conifer', getattr(conifer, '__version__', '?'),
      '| di_overdispersion default =', conifer.DiameterDistribution().di_overdispersion)

## 1. A synthetic population with known truth

Stem density by 6 DBH classes for `M` stands, generated from the compositional area-level Fay-Herriot
model: covariates drive the ALR composition, a random effect adds between-stand variation, and a
log-normal total sets the stems/acre. `S` is the true stems/acre-by-class we will try to recover.

In [ ]:
rng = np.random.default_rng(0)
M, K, p = 200, 6, 8                         # stands, DBH classes, covariates
X = rng.normal(size=(M, p))                 # LiDAR-like auxiliary covariates
B = rng.normal(scale=0.7, size=(p, K - 1))  # covariate -> ALR composition
alr = X @ B + rng.normal(scale=0.3, size=(M, K - 1))          # + between-stand random effect
full = np.hstack([alr, np.zeros((M, 1))]); full -= full.max(1, keepdims=True)
P = np.exp(full); P /= P.sum(1, keepdims=True)               # true composition (simplex)
N = rng.lognormal(mean=6.0, sigma=0.4, size=M)               # true total stems/acre
S = P * N[:, None]                                           # TRUE stems/acre by class
plot_lambda = N / 20.0                                       # ~stems per 1/20-acre plot
print('mean total stems/acre:', round(S.sum(1).mean(), 1), '| classes:', K)

## 2. Design-based sampler

For each stand we draw `k` plots (each a multinomial tally of trees), form the integer class **counts**
and the design-direct per-acre estimate, and the between-plot design variance of the log-total. This is the
same sampling model used in the full Monte-Carlo suite.

In [ ]:
NP_AVAIL, PA = 30, 1 / 20.0
def sample(k, seed):
    r = np.random.default_rng(seed)
    C = np.zeros((M, K)); sD = np.zeros((M, K)); lN = np.empty(M); vN = np.empty(M)
    for i in range(M):
        Tj = r.poisson(plot_lambda[i], NP_AVAIL)
        cj = np.array([r.multinomial(int(t), P[i]) for t in Tj]); dj = cj / PA
        kk = min(k, NP_AVAIL); sel = r.choice(NP_AVAIL, kk, replace=False)
        C[i] = cj[sel].sum(0); sD[i] = dj[sel].mean(0)
        tot = max(sD[i].sum(), 1e-3); lN[i] = np.log(tot)
        vN[i] = (max(dj[sel].sum(1).var(ddof=1), 1e-6) / (kk * tot ** 2)) if kk >= 2 else 0.25
    return C, sD, lN, vN
def rmse(s): return np.sqrt(np.mean((np.log1p(s) - np.log1p(S)) ** 2))

## 3. RMSE vs sampling intensity: CONIFER vs Direct vs k-NN

CONIFER should win at small `k` (borrowing strength) and converge onto Direct as `k` grows.

In [ ]:
from sklearn.neighbors import NearestNeighbors
def knn(Xz, sD, kk=7):
    Pd = sD / np.clip(sD.sum(1, keepdims=True), 1e-9, None)
    nn = NearestNeighbors(n_neighbors=min(kk + 1, M)).fit(Xz); _, idx = nn.kneighbors(Xz)
    return np.array([Pd[idx[i, 1:]].mean(0) * sD[i].sum() for i in range(M)])

Xz = (X - X.mean(0)) / (X.std(0) + 1e-9)
ks = [1, 2, 3, 5, 8, 15]; REPS = 30
res = {m: [] for m in ['CONIFER', 'Direct', 'kNN']}
for k in ks:
    acc = {m: [] for m in res}
    for rep in range(REPS):
        C, sD, lN, vN = sample(k, 100 + rep)
        fl = conifer.DiameterDistribution(boot_g3=15, seed=rep).fit(
            C, np.full(M, 1.0), X, total_logN=lN, var_logN=vN)
        acc['CONIFER'].append(rmse(fl.s_hat_)); acc['Direct'].append(rmse(sD)); acc['kNN'].append(rmse(knn(Xz, sD)))
    for m in res: res[m].append(np.mean(acc[m]))
    print(f'k={k:2d}  CONIFER {res["CONIFER"][-1]:.3f}  Direct {res["Direct"][-1]:.3f}  kNN {res["kNN"][-1]:.3f}')

plt.figure(figsize=(7, 4.3))
for m, c in [('CONIFER', '#3a7d44'), ('Direct', '#c77d3a'), ('kNN', '#8899aa')]:
    plt.plot(ks, res[m], 'o-', color=c, lw=2.2, label=m)
plt.xlabel('plots per stand (k)'); plt.ylabel('log-density RMSE')
plt.title('CONIFER wins in the sparse regime and converges to Direct (v0.2)')
plt.legend(); plt.tight_layout(); plt.show()

## 4. Uncertainty: conformal set vs naive Gaussian interval

Calibrate the design-aware conformal set on half the stands (against known truth) and measure joint coverage
on the held-out half, versus the analytic Gaussian interval. Target is 0.90.

In [ ]:
C, sD, lN, vN = sample(3, 7)
fl = conifer.DiameterDistribution(boot_g3=40, seed=0).fit(C, np.full(M, 1.0), X, total_logN=lN, var_logN=vN)
perm = np.random.default_rng(0).permutation(M); cal = np.sort(perm[:M // 2]); test = np.sort(perm[M // 2:])
fl.conformalize(S, cal, joint=True, alpha=0.10)
cov_conf = np.mean([bool(np.ravel(fl.joint_covered(S, np.array([i])))[0]) for i in test])
sd = np.sqrt(np.clip(fl.s_var_, 0, None)); z = 1.645
cov_gauss = np.mean(((S >= fl.s_hat_ - z * sd) & (S <= fl.s_hat_ + z * sd))[test])
print(f'joint conformal coverage: {cov_conf:.3f}   (target 0.90)')
print(f'Gaussian interval coverage: {cov_gauss:.3f}  (anti-conservative -> use conformal)')

## Takeaway

On known truth, CONIFER beats the direct estimator where small-area estimation is meant to help (few plots
per stand) and converges to it when data are plentiful — the v0.2 sampling-covariance refinement. Its
design-aware conformal set is calibrated near the nominal level, while the analytic Gaussian interval
under-covers. The full-scale, multi-population Monte-Carlo (8 archetypes + 3 real-covariate plasmode
populations, competitor slate, all metrics) lives in `PSAE_work/simulation/` (see `SIMULATION_DESIGN.md`).